## Chapter 7: Finetuning To Follow Instructions

In [1]:
# printing package versions used in the notebook
from importlib.metadata import version

pkgs = [
    "torch",
    "matplotlib",
    "numpy",
    "tiktoken",
    "tqdm",
    "tensorflow",        # for loading OpenAI's weights
]

for pkg in pkgs:
    print(f"{pkg} version: {version(pkg)}")

torch version: 2.9.1+cu130
matplotlib version: 3.10.7
numpy version: 2.3.5
tiktoken version: 0.12.0
tqdm version: 4.67.1
tensorflow version: 2.20.0


In [2]:
import torch

print(f"Is GPU available: {torch.cuda.is_available()}")

Is GPU available: True


### 7.1 Introduction to Instruction finetuning

- pretraining an LLM involves a training procedure where it learns to generate one word at a time
- Hence, a pretrained LLM is good at text completion, but it is not good at following instructions

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/02.webp" width=500px>

- The steps involved in instruction finetuning are summarized in the figure below

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/03.webp" width=500px>

### 7.2 Dataset preparation

#### 7.2.1 Downloading Dataset

In [3]:
import json
import os
import requests


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


# The same code using urllib package

"""
import urllib

def download_and_load_file(file_path, url):

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data
"""


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

dataset = download_and_load_file(file_path, url)
print("Number of entries in the dataset:", len(dataset))

Number of entries in the dataset: 1100


In [4]:
print(f"Sample from dataset: {dataset[10]}")
print(f"Sample from dataset: {dataset[50]}")

Sample from dataset: {'instruction': 'What is the contraction for "will not"?', 'input': '', 'output': 'The contraction for "will not" is "won\'t".'}
Sample from dataset: {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


#### 7.2.2 Preprocessing & batching the dataset

- Instruction finetuning is often referred to as "supervised instruction finetuning" because it involves training a model on a dataset where the input-output pairs are explicitly provided
- There are different ways to format the entries as inputs to the LLM; the figure below illustrates two example formats that were used for training the Alpaca (https://crfm.stanford.edu/2023/03/13/alpaca.html) and Phi-3 (https://arxiv.org/abs/2404.14219) LLMs, respectively

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/04.webp?2" width=500px>

In [5]:
# utility function to create prompt template 
def format_input(record):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{record["instruction"]}"
    )

    input_text = f"\n\n### Input:\n{record["input"]}" if record["input"] else ""

    return instruction_text + input_text

In [6]:
sample_input = dataset[50]

sample_formatted_input = format_input(sample_input)
desired_response = f"\n\n### Response:\n{sample_input["output"]}"

print(sample_formatted_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [7]:
sample_input = dataset[10]

sample_formatted_input = format_input(sample_input)
desired_response = f"\n\n### Response:\n{sample_input["output"]}"

print(sample_formatted_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is the contraction for "will not"?

### Response:
The contraction for "will not" is "won't".


In [8]:
# splitting dataset into train, test and validation sets
train_count = int(len(dataset) * 0.85)
test_count = int(len(dataset) * 0.10)
validation_count = len(dataset) - train_count - test_count

train_dataset = dataset[:train_count]
test_dataset = dataset[train_count:train_count + test_count]
validation_dataset = dataset[train_count + test_count:]

In [9]:
print("Training set length:", len(train_dataset))
print("Validation set length:", len(validation_dataset))
print("Test set length:", len(test_dataset))

Training set length: 935
Validation set length: 55
Test set length: 110


- The dataset is converted into batches by using the following steps:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/06.webp?1" width=500px>

- An `InstructionDataset` class that pre-tokenizes all inputs in the dataset

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/07.webp?1" width=500px>

In [10]:
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, input_data, tokenizer):
        self.data = input_data

        # Pre-tokenize texts
        self.encoded_texts = []
        for prompt in input_data:
            instruction_text = format_input(prompt)
            response_text = f"\n\n### Response:\n{prompt["output"]}"
            query = instruction_text + response_text
            self.encoded_texts.append(
                tokenizer.encode(query, allowed_special={"<|endoftext|>"})
            )
    
    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)

In [11]:
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

print(gpt2_tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


- The custom "collate" function pads the training examples in each batch to have the same length (but different batches can have different lengths)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/08.webp?1" width=500px>

- Collate function will be passed to dataloder which appliese the function to each batch in the dataloader

In [12]:
# function to generate input batch with padded tokens
def custom_collate_function_v1(
    batch, 
    pad_tokenid = 50256,
    device = "cpu"
):
    input_list = []

    # Find the longest sequence in the batch
    # and increase the max length by +1, which will add one extra
    # padding token below
    max_batch_length = max(len(item) + 1 for item in batch)

    # pad the batch with length as longest sequence length in the batch
    for item in batch:
        new_item = item.copy()
        new_item += [pad_tokenid]           # Add an <|endoftext|> token which denotes end of text generation
        padded_item = (
            new_item + [pad_tokenid] * (max_batch_length - len(new_item))
        )
        # Via padded[:-1], we remove the extra padded token
        # that has been added via the +1 setting in batch_max_length
        # (the extra padding token will be used for target item generation which is left shifted by 1 token)
        input_item = torch.tensor(padded_item[:-1])
        input_list.append(input_item)
    
    input_batch = torch.stack(input_list).to(device)

    return input_batch

In [13]:
sample_input1 = [0, 1, 2, 3, 4]
sample_input2 = [5, 6]
sample_input3 = [7, 8, 9]

sample_batch = [
    sample_input1,
    sample_input2,
    sample_input3,
]

print(f"sample batch: \n{sample_batch}")

sample batch: 
[[0, 1, 2, 3, 4], [5, 6], [7, 8, 9]]


In [14]:
print(f"output of collate function v1: \n{custom_collate_function_v1(sample_batch)}")

output of collate function v1: 
tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


- Above function only returned the inputs to the LLM; however, for LLM training, we also need the target values
- Similar to pretraining an LLM, the targets are the inputs shifted by 1 position to the right, so the LLM learns to predict the next token

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/10.webp?1" width=400px>

In [15]:
# function to generate input and target batch with padded tokens
def custom_collate_function_v2(
    batch, 
    pad_tokenid = 50256,
    device = "cpu"
):
    input_list, target_list = [], []

    # Find the longest sequence in the batch
    max_batch_length = max(len(item) + 1 for item in batch)

    # pad the batch with length as longest sequence length in the batch
    for item in batch:
        new_item = item.copy()
        new_item += [pad_tokenid]           # Add an <|endoftext|> token which denotes end of text generation
        padded_item = (
            new_item + [pad_tokenid] * (max_batch_length - len(new_item))
        )
        # In input (padded[:-1]), the extra padded token is removed
        # the extra padding token is used for target item which is left shifted by 1 token 
        # thus avoiding a random number/missing token added to target item while left shifting
        input_item = torch.tensor(padded_item[:-1])
        target_item = torch.tensor(padded_item[1:])

        input_list.append(input_item)
        target_list.append(target_item)
    
    input_batch = torch.stack(input_list).to(device)
    target_batch = torch.stack(target_list).to(device)


    return input_batch, target_batch

In [16]:
sample_input_tensor, sample_target_tensor = custom_collate_function_v2(sample_batch)

print(f"Input batch: \n{sample_input_tensor}\n")
print(f"target batch: \n{sample_target_tensor}\n")

Input batch: 
tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])

target batch: 
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])



- Next, an `ignore_index` value is introduced to replace all padding token IDs with a new value, the purpose of this `ignore_index` is that we can ignore padding values in the loss function
- This means that we replace the token IDs corresponding to `50256` with `-100` as illustrated below

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/12.webp?2" width=500px>

- In addition, `allowed_max_length` is introduced in case we want to limit the length of the samples, this will be useful while working with your own datasets that are longer than the 1024 token context size supported by the GPT-2 model

In [17]:
# function to generate input batch with padded tokens and target batch with masked padded tokens
def custom_collate_function(
    batch, 
    pad_tokenid = 50256,
    ignore_index = -100,
    allowed_max_length = None,
    device = "cpu"
):
    input_list, target_list = [], []

    # Find the longest sequence in the batch
    max_batch_length = max(len(item) + 1 for item in batch)

    # pad the batch with length as longest sequence length in the batch
    for item in batch:
        new_item = item.copy()
        new_item += [pad_tokenid]           # Add an <|endoftext|> token which denotes end of text generation
        padded_item = (
            new_item + [pad_tokenid] * (max_batch_length - len(new_item))
        )
        # In input (padded[:-1]), the extra padded token is removed
        # the extra padding token is used for target item which is left shifted by 1 token 
        # thus avoiding a random number/missing token added to target item while left shifting
        input_item = torch.tensor(padded_item[:-1])
        target_item = torch.tensor(padded_item[1:])

        # Replace all but the first padding tokens in targets by ignore_index
        mask = target_item == pad_tokenid           # provide a boolen list with True if toekn id is 50256 else False
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            target_item[indices[1:]] = ignore_index
        
        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            input_item = input_item[:allowed_max_length]
            target_item = target_item[:allowed_max_length]

        input_list.append(input_item)
        target_list.append(target_item)
    
    input_batch = torch.stack(input_list).to(device)
    target_batch = torch.stack(target_list).to(device)

    return input_batch, target_batch

In [18]:
sample_input_tensor, sample_target_tensor = custom_collate_function(sample_batch)

print(f"Input batch: \n{sample_input_tensor}\n")
print(f"target batch: \n{sample_target_tensor}\n")

Input batch: 
tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])

target batch: 
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])



- Let's see what this replacement by -100 accomplishes
- For illustration purposes, let's assume we have a small classification task with 2 class labels, 0 and 1
- If we have the following logits values (outputs of the last layer of the model), we calculate the following loss

In [19]:
sample_logits = torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5]]  # 2nd training example
)
sample_targets = torch.tensor([0, 1])

sample_loss1 = torch.nn.functional.cross_entropy(sample_logits, sample_targets)
print(sample_loss1)

tensor(1.1269)


In [20]:
# adding one more training sample for loss calculation
sample_logits2 = torch.tensor(
   [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # New 3rd training example
)
sample_targets2 = torch.tensor([0, 1, 1])

sample_loss2 = torch.nn.functional.cross_entropy(sample_logits2, sample_targets2)
print(sample_loss2)

tensor(0.7936)


In [21]:
# now replacing target output with default ignore value(-100) of cross entropy loss
sample_targets3 = torch.tensor([0, 1, -100])

sample_loss3 = torch.nn.functional.cross_entropy(sample_logits2, sample_targets3)
print(sample_loss3)
print("loss_1 == loss_3:", sample_loss1 == sample_loss3)

tensor(1.1269)
loss_1 == loss_3: tensor(True)


- The resulting loss on these 3 training examples from sample loss 3 is the same as the loss we calculated from the 2 training examples in sample loss 1, which means that the cross-entropy loss function ignored the training example with the -100 label
- By default, PyTorch has the `cross_entropy(..., ignore_index=-100)` setting to ignore examples corresponding to the label -100
- Using this -100 `ignore_index`, we can ignore the additional end-of-text (padding) tokens in the batches that we used to pad the training examples to equal length
- However, we don't want to ignore the first instance of the end-of-text (padding) token (50256) because it can help signal to the LLM when the response is complete

- In practice, it is also common to mask out the target token IDs that correspond to the instruction, as illustrated in the figure below 

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/13.webp" width=600px>

#### 7.2.3 Creating Dataloaders

- The `custom_collate_fn` function can be set to directly move the data to the target device (e.g., GPU) instead of doing it in the main training loop, which improves efficiency because it can be carried out as a background process when we use the `custom_collate_fn` as part of the data loader
- Using the `partial` function from Python's `functools` standard library, we create a new function with the `device` argument(other arguments needed) of the original function pre-filled

In [22]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: cuda


In [23]:
from functools import partial

customized_collate_function = partial(
    custom_collate_function,
    device = device,
    allowed_max_length = 1024
)

In [24]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

# training dataloader
torch.manual_seed(123)
instruction_train_dataset = InstructionDataset(train_dataset, gpt2_tokenizer)

train_dataloader = DataLoader(
    dataset=instruction_train_dataset,
    collate_fn=customized_collate_function,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [25]:
# validation dataloader
instruction_validation_dataset = InstructionDataset(validation_dataset, gpt2_tokenizer)

validation_dataloader = DataLoader(
    dataset=instruction_validation_dataset,
    collate_fn=customized_collate_function,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

# test dataloader
instruction_test_dataset = InstructionDataset(test_dataset, gpt2_tokenizer)

test_dataloader = DataLoader(
    dataset=instruction_test_dataset,
    collate_fn=customized_collate_function,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [26]:
print("Train loader:")
for inputs, targets in train_dataloader:
    print(inputs.shape, targets.shape)

Train loader:
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 68]) torch.

- As we can see based on the output above, all batches have a batch size of 8 but a different length, as expected
- Let's also double-check that the inputs contain the `<|endoftext|>` padding tokens corresponding to token ID 50256 by printing the contents of the first training example in the `inputs` batch
- Similary, check targets contain padding tokens that are masked by -100

In [27]:
print(f"Input: \n{inputs[0]}\n")
print(f"Target: \n{targets[0]}\n")

Input: 
tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
          257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
        21017, 46486,    25,   198, 30003,  6525,   262,  6827,  1262,   257,
          985,   576,    13,   198,   198, 21017, 23412,    25,   198,   464,
         5156,   318,   845, 13779,    13,   198,   198, 21017, 18261,    25,
          198,   464,  5156,   318,   355, 13779,   355,   257,  4936,    13,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256],
       device='cuda:0')

Target: 
tensor([  318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,   257,
         2882,   326, 20431, 32543,   262,  2581,    13,   198,   198, 21017,
        46486,    25,   198, 30003,  6525,   262,  6827,  1262,   257,   985,
          576,    13,   198,   198, 21017, 23412,    25,   198,   464,  5156,
          318,   845, 13779,    13,   198,   198, 21017, 18261,    25,   198,
          464,  5156,   318,

### 7.5 Loading Pre-trained LLM

In [28]:
GPT2_BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "contextWindow_len": 1024,  # Context length
    "dropout_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"embedding_dim": 768, "num_transformerLayers": 12, "num_heads": 12},
    "gpt2-medium (355M)": {"embedding_dim": 1024, "num_transformerLayers": 24, "num_heads": 16},
    "gpt2-large (774M)": {"embedding_dim": 1280, "num_transformerLayers": 36, "num_heads": 20},
    "gpt2-xl (1558M)": {"embedding_dim": 1600, "num_transformerLayers": 48, "num_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"

GPT2_BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

print(f"{GPT2_BASE_CONFIG}")

{'vocab_size': 50257, 'contextWindow_len': 1024, 'dropout_rate': 0.0, 'qkv_bias': True, 'embedding_dim': 1024, 'num_transformerLayers': 24, 'num_heads': 16}


In [29]:
from openai_gpt2_download import download_and_load_gpt2

# get model size from the model name
gpt2_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=gpt2_size, models_dir="..\\Ch05\\gpt2")

print(f"\nGPT2-{gpt2_size} loaded with configs as: \n{settings}\n{params.keys()}")

File already exists and is up-to-date: ..\Ch05\gpt2\355M\checkpoint
File already exists and is up-to-date: ..\Ch05\gpt2\355M\encoder.json
File already exists and is up-to-date: ..\Ch05\gpt2\355M\hparams.json
File already exists and is up-to-date: ..\Ch05\gpt2\355M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: ..\Ch05\gpt2\355M\model.ckpt.index
File already exists and is up-to-date: ..\Ch05\gpt2\355M\model.ckpt.meta
File already exists and is up-to-date: ..\Ch05\gpt2\355M\vocab.bpe

GPT2-355M loaded with configs as: 
{'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 1024, 'n_head': 16, 'n_layer': 24}
dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])


In [30]:
torch.manual_seed(123)

input_text = format_input(validation_dataset[0])
print(input_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'


In [31]:
from previous_chapters import (
    GPTModel, 
    load_weights_into_gpt_instance, 
    generate_tokens_with_sampling, 
    text_to_token_ids, 
    token_ids_to_text
)

gpt2_355m = GPTModel(GPT2_BASE_CONFIG)

# loading pre-trained weights into model instance
load_weights_into_gpt_instance(gpt2_355m, params)
gpt2_355m.eval();

model_output = generate_tokens_with_sampling(
    model=gpt2_355m,
    input_ids=text_to_token_ids(input_sentence=input_text, tokenizer=gpt2_tokenizer),
    context_window=GPT2_BASE_CONFIG["contextWindow_len"],
    max_new_tokens=25,
    eos_id=50256
)

generated_text = token_ids_to_text(token_ids=model_output, tokenizer=gpt2_tokenizer)
print(f"GPT2 model ouput: \n{generated_text}")

GPT2 model ouput: 
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'

### Response:

The chef cooks the meal every day.

### Instruction:

Convert the


In [32]:
response_text = (
    generated_text[len(input_text):]
    .replace("### Response:", "")
    .strip()
)

print(response_text)

The chef cooks the meal every day.

### Instruction:

Convert the


### 7.6 Finetuning LLM on instruction data

In [33]:
from previous_chapters import (
    model_training_simple,
    calc_loss_dataloader,
    plot_losses
)

In [34]:
gpt2_355m.to(device)

torch.manual_seed(123)

with torch.no_grad():
    training_loss = calc_loss_dataloader(train_dataloader, gpt2_355m, device, num_batches=5)
    validation_loss = calc_loss_dataloader(validation_dataloader, gpt2_355m, device, num_batches=5)

print(f"The initial training loss on sample(n-batches:5): {training_loss}")
print(f"The initial validation loss on sample(n-batches:5): {validation_loss}")

The initial training loss on sample(n-batches:5): 3.8259104251861573
The initial validation loss on sample(n-batches:5): 3.7619346141815186


In [35]:
# use CPU if GPU is out of memory 
# device = torch.device("cpu")
# print(f"using device: {device}")

In [36]:
# estimating time for training LLM
import time
start_time = time.time()

# Initiate optimizer
torch.manual_seed(123)
gpt2_355m.to(device)
adam_optimizer = torch.optim.AdamW(gpt2_355m.parameters(), lr = 0.00005, weight_decay = 0.1)

n_epochs = 2
# Initiate a LLM training 
sample_training_losses, sample_validation_losses, tokens_seen = model_training_simple(
    model = gpt2_355m,
    train_dataloader = train_dataloader,
    validation_dataloader = validation_dataloader,
    optimizer = adam_optimizer,
    device = device,
    num_epochs = n_epochs,
    eval_frequency = 5,
    eval_batch_iteration = 5,
    sample_input = format_input(validation_dataset[0]),
    tokenizer = gpt2_tokenizer,
    max_tokens = 50
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"\nTraining completed in {execution_time_minutes:.2f} minutes.")

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 6.69 GiB is allocated by PyTorch, and 96.81 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)